In [2]:
import os
import sys
import json
import httpx
import sqlite3
import gradio as gr

from pathlib import Path
from openai import OpenAI

parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir))
from config import CONFIG


In [16]:
llm_config = CONFIG['llm']
models_config = CONFIG['models']

openai = OpenAI(
    base_url=llm_config['llm_base_url'],
    api_key=llm_config['llm_api_key'],
    http_client=httpx.Client(verify=False, timeout=60)
)

In [18]:
# check without tools is it can check the ticket price or not
def check_price_without_tools():
    message = [
        {"role": "system", "content": "You are very helpful assistant. Please answer it short, precise and concise. If you do not know, please so."},
        {"role": "user", "content": "can you help to check ticket price return from london to tokyo?"}
        ]
    response = openai.chat.completions.create(model=models_config['gpt'], messages=message)
    return response.choices[0].message.content

In [19]:
check_price_without_tools()

'I can help — I can’t fetch live fares, but I can either guide you on where to check or estimate typical prices and search tips. Quick questions first:\n\n- Exact travel dates (or flexible?), one-way or return?  \n- Number of passengers and cabin (economy, premium econ, business)?  \n- Depart from which London airport (LHR, LGW, STN, etc.)?  \n- Prefer direct flights or OK with stopovers?  \n- Currency you want prices in?\n\nTypical return price ranges (approximate, can vary):\n- Economy: £500–£1,200 (off-peak → peak)  \n- Premium economy: £1,200–£2,000  \n- Business: £2,800–£5,500+\n\nBest sites to check: Google Flights, Skyscanner, Kayak, Momondo, or airline sites (British Airways, ANA, Japan Airlines). Tips: be flexible with dates, set price alerts, compare nearby airports, search in incognito, book 2–6 months ahead for best deals.\n\nGive me the details above and I’ll suggest specific options and search steps.'

In [22]:
class FlightTicketPrice:

    def __init__(self, db_path="prices.db"):
        self.db_path = db_path
        self.tools = [
            {
                "type": "function", 
                "function": {
                                "name": "set_ticket_price",
                                "description": "Set the price of a return ticket to the destination city.",
                                "parameters": {
                                    "type": "object",
                                    "properties": {
                                        "city": {
                                            "type": "string",
                                            "description": "The city that the customer wants to set",
                                        },
                                        "price": {
                                            "type": "number",
                                            "description": "The price to be setup"
                                        },
                                    },
                                    "required": ["city", "price"],
                                    "additionalProperties": False
                                }
                            }
                }
            ]
        self.system_message = """
                                You are a helpful assistant for an Airline called FlightAI.
                                Give short, courteous answers, no more than 1 sentence.
                                Always be accurate. If you don't know the answer, say so.
                            """

    def get_ticket_price(self, city):
        with sqlite3.connect(self.db_path) as conn:
            cursor = conn.cursor()
            cursor.execute(f"""SELECT prices FROM prices WHERE city = {city};""")
            result = cursor.fetchone()

            return f"The ticket price to {city}: {result[0]}$" if result else f"The ticket price to this {city} is not available right now!"
        
    def set_ticket_price(self, city, price):
        if price < 0:
            return "The price must be greater than zero."
        
        with sqlite3.connect(self.db_path) as conn:
            cur = conn.cursor()
            cur.execute(
                """
                INSERT INTO prices (city, price)
                VALUES(?, ?)
                ON CONFLICT(city) DO UPDATE SET price = excluded.price
                """, (city, price)
                )
            
            return f"The ticket price for {city} was set to ${price:.2f}."

    def handle_tool_call(self, tool_call):
        if tool_call.function.name == "set_ticet_price":
            arguments = json.loads(tool_call.function.arguments)
            print("--- CHECKING ARGUMENTS ---")
            return self.set_ticket_price(arguments['city'], arguments['price'])
        return "Unsupported tool"


    def chat(self, user_message, history):
        messages = [
            {"role": h["role"], "content": h["content"]} for h in history
            ]
        
        messages = [{"role": "system", "content": self.system_message}] + messages + [{"role": "user", "content": user_message}]

        response = openai.chat.completions.create(
            model=models_config['gpt'], 
            messages=messages, 
            tools=self.tools
            )

        assistant_message = response.choices[0].message
        if assistant_message.tool_calls:
            # Add assistant's request first.
            messages.append(assistant_message)

            tool_call = assistant_message.tool_calls[0]
            print("--- TOOL CALL RESULTS ---")
            print(tool_call)
            tool_result = self.handle_tool_call(tool_call)
            print("--- FINAL TOOL CALL RESULT ---")
            print(tool_result)

            # Then add the result for that specific tool call.
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result
                }
            )

            response = openai.chat.completions.create(
                model=models_config['gpt'], messages=messages
                )
        return response.choices[0].message.content


In [27]:
flight_ticket_price = FlightTicketPrice()
flight_ticket_price.chat("Set the return ticket price for Jakarta to 600.", history=[])

--- TOOL CALL RESULTS ---
ChatCompletionMessageFunctionToolCall(id='call_ojnHScuJtUS8i4EV1ZgU2cJN', function=Function(arguments='{"city":"Jakarta","price":600}', name='set_ticket_price'), type='function')
--- FINAL TOOL CALL RESULT ---
Unsupported tool


'I’m unable to change ticket prices directly—please contact your admin or use the pricing dashboard to set the Jakarta return fare to 600.'